# Day 7：Week 1 整合与复盘

今天是 Week 1 的最后一天，**不学新东西**。

目标是把本周写的代码整理好，构建一个完整的命令行 Agent，跑通"思考→调用→检查→纠错"的完整闭环。

## Week 1 知识回顾

```
Day 1-2: StateGraph 管理流程    → 节点、边、条件路由、中断
Day 3:   Tool Calling 鲁棒性    → Pydantic Schema、中间件、降级策略
Day 4:   MCP 协议入门          → 工具标准化、动态发现
Day 5-6: Context & Memory 管理 → 滑动窗口、摘要压缩、Checkpointer
         ↓
Day 7:   整合成一个工业级 Agent
```

这四个组件单独每个都不够，**组合在一起才是一个"工业级"Agent 的基础**。

## 一、完整 Agent 架构设计

```
┌─────────────────────────────────────────────────────────┐
│                    Week 1 Agent                          │
│                                                          │
│  用户输入                                                 │
│    ↓                                                     │
│  ┌─────────┐    ┌──────────┐    ┌─────────┐             │
│  │ Memory  │ →  │   LLM    │ →  │  Tools  │             │
│  │ 模块    │    │ (思考)    │    │ (执行)   │             │
│  └─────────┘    └──────────┘    └─────────┘             │
│       ↑              │               │                   │
│       │              │ 条件路由      │                    │
│       │              ↓               ↓                    │
│       │         ┌──────────┐    ┌─────────┐             │
│       └─────────│ 中间件   │←───│ 结果返回 │             │
│                 └──────────┘    └─────────┘             │
│                      ↓                                    │
│                 格式化/日志/纠错                           │
└─────────────────────────────────────────────────────────┘
```

**核心能力：**
- StateGraph 管理复杂状态流转
- 工具调用带中间件（参数校验、超时、重试）
- Memory 模块维持长对话
- 条件路由实现重试与错误恢复

## 二、代码整合：构建 Week 1 Agent

把所有组件整合到一个完整的 Agent 中。

In [ ]:
import os
import json
import subprocess
import sys
import time
from typing import TypedDict, Annotated, Literal, Optional
from pathlib import Path

from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command
from langchain_openai import ChatOpenAI
from langchain_core.messages import (
    HumanMessage, AIMessage, ToolMessage, SystemMessage, AnyMessage
)


# =====================================================
# 第一部分：工具 Schema 定义（复用 Day 3 的内容）
# =====================================================

class FileReadInput(BaseModel):
    path: str = Field(description="文件的绝对路径")
    encoding: str = Field(default="utf-8", description="文件编码")
    max_lines: Optional[int] = Field(default=None, description="最多读取行数")

class ShellExecInput(BaseModel):
    command: str = Field(description="Shell 命令")
    timeout: int = Field(default=30, description="超时时间（秒）")

class PythonRunInput(BaseModel):
    code: str = Field(description="Python 代码")
    timeout: int = Field(default=30, description="超时时间（秒）")


# =====================================================
# 第二部分：LLM 初始化
# =====================================================

llm = ChatOpenAI(
    model="mimo-v2.5-pro",
    api_key=os.getenv("MIMO_API_KEY"),
    base_url="https://token-plan-cn.xiaomimimo.com/v1",
    temperature=0,
)
llm_with_tools = llm.bind_tools([FileReadInput, ShellExecInput, PythonRunInput])


# =====================================================
# 第三部分：状态定义
# =====================================================

class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    retry_count: int       # 工具调用重试次数
    memory_summary: str    # 对话摘要（Day 5-6 的内容）
    turn_count: int        # 对话轮数


# =====================================================
# 第四部分：工具执行（带中间件，复用 Day 3 的内容）
# =====================================================

def tool_file_read(path: str, encoding: str = "utf-8", max_lines: int = None) -> str:
    """读取文件"""
    file_path = Path(path)
    if not file_path.exists():
        return f"错误：文件不存在: {path}"
    with open(file_path, "r", encoding=encoding) as f:
        if max_lines:
            return "".join([next(f) for _ in range(max_lines)])
        return f.read()

def tool_shell_exec(command: str, timeout: int = 30) -> dict:
    """执行 Shell 命令"""
    try:
        result = subprocess.run(
            command, shell=True, capture_output=True, text=True, timeout=timeout
        )
        return {
            "success": True,
            "stdout": result.stdout[:2000],
            "stderr": result.stderr[:2000],
            "return_code": result.returncode
        }
    except subprocess.TimeoutExpired:
        return {"success": False, "error": f"命令超时（>{timeout}s）", "error_type": "timeout"}
    except Exception as e:
        return {"success": False, "error": str(e), "error_type": type(e).__name__}

def tool_python_run(code: str, timeout: int = 30) -> dict:
    """运行 Python 代码"""
    try:
        result = subprocess.run(
            [sys.executable, "-c", code],
            capture_output=True, text=True, timeout=timeout
        )
        return {
            "success": True,
            "stdout": result.stdout[:3000],
            "stderr": result.stderr[:3000],
            "return_code": result.returncode
        }
    except subprocess.TimeoutExpired:
        return {"success": False, "error": f"代码执行超时（>{timeout}s）", "error_type": "timeout"}
    except Exception as e:
        return {"success": False, "error": str(e), "error_type": type(e).__name__}

# 工具注册表
TOOLS = {
    "file_read": tool_file_read,
    "shell_exec": tool_shell_exec,
    "python_run": tool_python_run,
}


# =====================================================
# 第五部分：节点函数
# =====================================================

def call_llm(state: AgentState):
    """
    LLM 思考节点——Agent 的"大脑"
    
    逻辑：
    1. 如果有 Memory 摘要，注入 system message
    2. 调用 LLM（带工具绑定）
    3. LLM 返回：回答文本 或 工具调用请求
    """
    messages = list(state["messages"])
    
    # 利用 Memory 摘要（Day 5-6 学的内容）
    if state.get("memory_summary"):
        summary = state["memory_summary"]
        messages = [
            SystemMessage(content=f"之前对话的关键信息总结：{summary}")
        ] + messages
    
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}


def execute_tools(state: AgentState):
    """
    工具执行节点——Agent 的"手"
    
    逻辑：
    1. 获取 LLM 请求的工具调用
    2. 找到对应的工具函数并执行
    3. 把结果包装成 ToolMessage 返回
    """
    last_msg = state["messages"][-1]
    tool_messages = []
    retry = state.get("retry_count", 0)
    
    for tool_call in last_msg.tool_calls:
        name = tool_call["name"]
        args = tool_call["args"]
        func = TOOLS.get(name)
        
        if func is None:
            result_str = f"错误：未知工具 '{name}'。可用工具：{list(TOOLS.keys())}"
        else:
            try:
                result = func(**args)
                result_str = json.dumps(result, ensure_ascii=False)
            except Exception as e:
                result_str = f"工具执行失败: {str(e)}"
        
        tool_messages.append(ToolMessage(
            content=result_str,
            tool_call_id=tool_call["id"]
        ))
    
    return {
        "messages": tool_messages,
        "retry_count": retry + 1
    }


def manage_memory(state: AgentState):
    """
    Memory 管理节点——防止上下文溢出
    
    逻辑：每 5 轮生成一次对话摘要（Day 5-6 的内容）
    """
    turn = state.get("turn_count", 0) + 1
    
    if turn > 0 and turn % 5 == 0:
        messages = state["messages"]
        recent = messages[-10:]
        text = "\n".join([
            f"{'用户' if isinstance(m, HumanMessage) else 'AI'}: {m.content[:80]}"
            for m in recent
        ])
        
        existing = state.get("memory_summary", "")
        prompt = f"""把以下新对话合并进历史摘要：

历史摘要：{existing or '（对话开始）'}

新对话：
{text}

输出更新后的摘要（3-5 句话）："""
        
        new_summary = llm.invoke(prompt).content
        return {"memory_summary": new_summary, "turn_count": turn}
    
    return {"turn_count": turn}


# =====================================================
# 第六部分：条件路由——Agent 的"神经系统"
# =====================================================

def route_after_llm(state: AgentState) -> Literal["tools", "memory", "end"]:
    """
    决定 LLM 输出后走哪条路径
    
    - 有工具调用 → tools 节点
    - 无工具调用 → memory 节点（更新记忆）→ end
    
    但如果重试超过 3 次，强制结束（防止无限循环）
    """
    last_msg = state["messages"][-1]
    
    if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
        if state.get("retry_count", 0) >= 3:
            return "end"  # 重试太多，停止
        return "tools"
    
    return "memory"


# =====================================================
# 第七部分：构建状态图
# =====================================================

builder = StateGraph(AgentState)

# 添加节点
builder.add_node("llm", call_llm)
builder.add_node("tools", execute_tools)
builder.add_node("memory", manage_memory)

# 添加边
builder.add_edge(START, "llm")

# 条件边：LLM 输出决定去向
builder.add_conditional_edges(
    "llm", route_after_llm,
    {
        "tools": "tools",
        "memory": "memory",
        "end": END
    }
)

# 工具执行完回到 LLM（形成 ReAct 循环）
builder.add_edge("tools", "llm")

# Memory 更新完后结束
builder.add_edge("memory", END)

# 编译
agent = builder.compile(checkpointer=InMemorySaver())

print("Week 1 Agent 构建完成！")
print(f"节点: {list(builder.builder._nodes.keys())}")
print(f"工具: {list(TOOLS.keys())}")
print(f"\n完整闭环: START → LLM ↔ Tools (ReAct) → Memory → END")

## 三、测试：完整闭环

测试 Agent 的"思考→调用→检查→纠错"完整闭环。

In [ ]:
# ===== 测试 1：基础对话 =====
print("=" * 50)
print("测试 1：基础对话")
print("=" * 50)

config = {"configurable": {"thread_id": "test-1"}}

result = agent.invoke({
    "messages": [HumanMessage(content="你好，用 Python 帮我计算 1+2+3+...+100 的和")],
    "retry_count": 0,
    "turn_count": 0,
    "memory_summary": ""
}, config)

for msg in result["messages"]:
    role = type(msg).__name__.replace("Message", "")
    content = msg.content if hasattr(msg, "content") else str(msg)
    print(f"[{role}] {str(content)[:100]}")

print(f"\n总步数: {result.get('retry_count', 'N/A')}")

In [ ]:
# ===== 测试 2：工具调用失败后的重试（核心测试） =====
print("=" * 50)
print("测试 2：阅读不存在的文件（测试错误处理）")
print("=" * 50)

config = {"configurable": {"thread_id": "test-2"}}

result = agent.invoke({
    "messages": [HumanMessage(content="请读取文件 /nonexistent/file.txt 的内容")],
    "retry_count": 0,
    "turn_count": 0,
    "memory_summary": ""
}, config)

for msg in result["messages"]:
    role = type(msg).__name__.replace("Message", "")
    content = msg.content if hasattr(msg, "content") else str(msg)
    print(f"[{role}] {str(content)[:120]}")

print(f"\n重试次数: {result.get('retry_count', 0)}")

In [ ]:
# ===== 测试 3：Memory 长对话 =====
print("=" * 50)
print("测试 3：Memory 长对话")
print("=" * 50)

config = {"configurable": {"thread_id": "test-3"}}

# 先告诉 Agent 个人信息
init_state = {
    "messages": [HumanMessage(content="我叫张三，今年25岁，住在北京")],
    "retry_count": 0,
    "turn_count": 0,
    "memory_summary": ""
}

for i in range(7):  # 7 轮对话
    if i == 0:
        result = agent.invoke(init_state, config)
    elif i == 6:
        # 最后一轮：测试记忆
        result = agent.invoke(
            {"messages": [HumanMessage(content="你还记得我叫什么吗？我多大了？住在哪里？")]},
            config
        )
    else:
        result = agent.invoke(
            {"messages": [HumanMessage(content=f"这是第 {i+1} 轮测试消息")]},
            config
        )
    
    if i == 6:
        print(f"\n第 {i+1} 轮（记忆测试）:")
        print(f"Agent: {result['messages'][-1].content[:200]}")

# 查看记忆状态
final = agent.get_state(config)
summary = final.values.get("memory_summary", "")
print(f"\n对话摘要: {summary[:200] if summary else '未生成摘要'}")

## 四、Agent 的边界与失败模式

一个好的 Agent 工程师不仅要知道 Agent 能做什么，更要知道**Agent 会在哪里失败**。

### 常见失败模式

| 失败模式 | 原因 | 当前处理 |
|---------|------|---------|
| LLM 幻觉调用不存在的工具 | LLM 生成了一个不在工具列表里的工具名 | TOOLS 字典兜底，返回"未知工具"提示 |
| 无限循环 | LLM 反复执行同一个失败操作 | retry_count 限制，超过 3 次强制退出 |
| 上下文溢出 | 长对话后 State 膨胀超出 Token 限制 | Memory 管理节点，每 5 轮生成摘要 |
| 工具执行超时 | 网络问题/资源问题导致工具卡住 | subprocess timeout 保护 |
| 权限问题 | Agent 尝试执行无权限的操作 | subprocess 的 cwd 和环境变量限制 |

### Agent 设计的核心原则

经过 Week 1 的学习和实践，总结三条核心原则：

**1. 最小权限原则**
Agent 只能访问必要的工具和数据。不要给它 root 权限，不要让它直接操作数据库。
每个工具都要限制执行范围（如 file_read 只能读指定目录）。

**2. 可观测性原则**
每一步都要有日志，失败时能回溯原因。
工具调用记录（中间件的历史记录）、LLM 的输入/输出、状态变化——都要能追踪。

**3. 优雅降级原则**
主路径失败时有备选方案。
工具调用失败 → 重试 → 超过上限 → 告知用户 → 询问是否继续。
永远不要让 Agent 静默失败。

## 五、GitHub 仓库准备

整理本周代码，为简历项目做准备。

In [ ]:
# ===== 导出 Agent 核心代码到独立模块 =====
# 这是你 GitHub 项目的基础

WEEK1_MODULE = '''\"\"\"
Week 1 Agent：基于 LangGraph 的工业级 Agent

核心能力：
- StateGraph 管理复杂状态流转
- 工具调用带中间件（参数校验、超时、重试）
- Memory 模块维持长对话
- 条件路由实现重试与错误恢复

使用方式：
    from agent import Week1Agent
    agent = Week1Agent()
    result = agent.run("你的任务描述")
\"\"\"

# 实际项目中，把上面的代码组织成清晰的模块：
# agent/
#   __init__.py
#   state.py      —— AgentState 定义
#   tools.py      —— 工具函数 + Schema
#   nodes.py      —— LLM、工具执行、Memory 节点
#   router.py     —— 条件路由函数
#   graph.py      —— 状态图构建
#   main.py       —— 入口 + Human-in-the-loop 循环
'''

print(WEEK1_MODULE)

## 今日总结

今天把 Week 1 的四天知识整合成了一个完整的 Agent：

| 组件 | 来源 | 作用 |
|------|------|------|
| StateGraph | Day 1-2 | 管理状态流转，支持条件路由和循环 |
| 工具 + 中间件 | Day 3 | Pydantic Schema、超时、重试、错误处理 |
| Memory 管理 | Day 5-6 | 滑动窗口 + 摘要压缩，防止上下文溢出 |
| 完整闭环 | Day 7 | 把以上组件串联成"思考→调用→检查→纠错" |

**Week 1 周末里程碑达成：**
- [x] 手写一个含状态流转的命令行 Agent
- [x] 支持工具调用失败后自动重试
- [x] 能跑通"思考→调用→检查→纠错"完整闭环
- [x] 在 GitHub 建好项目仓库，写好基础 README

下周学习高级 RAG 和推理加速原理——为简历增加算法护城河。